# HW3: Model Predictive Control
# Part 1: Planar drone MPC and obstacle avoidance using SQP
Ting-Wei Hsu (twhsu3)

In [ ]:
import numpy as np
from qpsolvers import solve_problem, Problem
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Suppress the display of very small numbers
np.set_printoptions(suppress=True)


## Dynamical System

Consider a planar drone with the equations of motion
$$
\begin{aligned}
\dot{p}_x &= v_x \\
\dot{p}_z &= v_z \\
\dot{\theta} &= \omega \\
m\dot{v}_x &= -(T_1 + T_2)\sin\theta \\
m\dot{v}_z &= (T_1 + T_2)\cos\theta - mg \\
J_y \dot{\omega} &= r(T_1 - T_2)
\end{aligned}
$$

where $m = 0.45$ is the mass; 
$r = 0.14$ is the spar length;
$J_y = 0.09$ is the moment of inertia about the $y$ (out-of-plane) axis;
$g = 9.81$ is the acceleration of gravity;
$p_x$ and $p_z$ are the components of position;
$\theta$ is the orientation;
$v_x$ and $v_z$ are the components of linear velocity;
$\omega$ is the angular velocity;
$T_1$ and $T_2$ are the forces produced by each rotor.

We define the state and input as
$$
x = [p_x,\; p_z,\; \theta,\; v_x,\; v_z,\; \omega]^\top,
 \qquad
u = [T_1,\; T_2]^\top
$$
and the dynamicas as 
$$
\dot{x} = f(x,u)
$$

In [ ]:
m = 0.45
r = 0.14
Jy = 0.09
grav = 9.81
dt = 0.04
N = 25
sim_time = 5.0

x_dim = 6
u_dim = 2

def dynamics(x, u):
    px, pz, theta, vx, vz, omega = x
    T1, T2 = u
    return np.array([
        vx,
        vz,
        omega,
        -(T1 + T2) / m * np.sin(theta),
        (T1 + T2) / m * np.cos(theta) - grav,
        r * (T1 - T2) / Jy,
    ])


def Jacobian_dynamics(x, u):
    _, _, theta, _, _, _ = x
    T1, T2 = u

    A = np.zeros((x_dim, x_dim))
    A[0, 3] = 1.0
    A[1, 4] = 1.0
    A[2, 5] = 1.0
    A[3, 2] = -(T1 + T2) / m * np.cos(theta)
    A[4, 2] = -(T1 + T2) / m * np.sin(theta)

    B = np.zeros((x_dim, u_dim))
    B[3, :] = -(1.0 / m) * np.sin(theta)
    B[4, :] =  (1.0 / m) * np.cos(theta)
    B[5, 0] =  r / Jy
    B[5, 1] = -r / Jy
    return A, B


## Problem Formulation
The trajectory optimization problem with obstacle avoidance is formulated as the following optimal control problem:
$$
\begin{align}
\min_{\substack{x_0,\cdots x_N \\ u_0,\cdots u_{N-1}}}& \sum_{k=0}^{N-1} \frac{1}{2}(x_k - x_{\mathrm{ref},k})^\top Q (x_k - x_{\mathrm{ref},k}) + \frac{1}{2}(u_k - u_{\mathrm{ref},k})^\top R (u_k - u_{\mathrm{ref},k}) + \frac{1}{2}(x_N - x_{\mathrm{goal}})^\top Q_F (x_N - x_{\mathrm{goal}})\\

\text{s.t.} & \quad x_0 = x_{\mathrm{start}} \\
& \quad x_{k+1} - x_k - f(x_k, u_k) \Delta t = 0 \quad \forall k = 0,1,\cdots N-1 \\
& \quad (2r)^2 - (p_{x,k} - p_{\mathrm{obs},x})^2 - (p_{z,k} - p_{\mathrm{obs},z})^2 \leq 0

\end{align}
$$
where $J$ denotes the cost function, the decision variables are
- $x_0, \dots, x_N$ (with $x_0 = x_{\mathrm{start}}$)
- $u_0, \dots, u_{N-1}$

the dynamics constraint is encoded by the equality constraint in (3), and the obstacle avoidance constraint is encoded by the inequality constraint in (4).

In this homework, we consider 
- $N = 25$
- $\Delta t = 0.04$
- $x_{\mathrm{start}} = [0,0,0,0,0,0]^\top$ 
- $x_{\mathrm{goal}} = [3,0,0,0,0,0]^\top$
- $x_{\mathrm{ref},k} = x_{\mathrm{goal}} \quad \forall k=0,1,\cdots N$
- $u_{\mathrm{ref},k} = [mg/2, mg/2]^\top \quad \forall k=0,1,\cdots N$

The position of the obstacle $(p_{\mathrm{obs},x}, p_{\mathrm{obs},z})$ will be determined later.


For simplicity of notation, we will rewrite the optimal control problem using the following notation

$$
\begin{align*}
& \min_{\mathrm{z}} J(\mathrm{z})\\

\text{s.t.} &\quad g(\mathrm{z}) = 0 \\
&\quad h(\mathrm{z}) \leq 0,
\end{align*}
$$
where $\mathrm{z}=[x_0, \dots, x_N, u_0, \dots, u_{N-1}]^\top$ denote the decisiton variables, $g(\mathrm{z})$ denotes the equality constraint, and $h(\mathrm{z})$ denotes the inequality constraint.

Define the **Lagrangian function**

$$ L(\mathrm{z},\lambda, \nu) = J(\mathrm{z}) + \lambda^\top g(\mathrm{z}) + \nu^\top h(\mathrm{z})$$

and compute both its gradient $\nabla_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu)$ and its hessian $\mathbf{H}_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu)$ with respect to $\mathrm{z}$.

The optimization problem can be solved by **sequential quadratic programming (SQP)**, where the approximated QP problem is given by
$$
\begin{align*}
 \min_{\Delta\mathrm{z}}& \quad (\nabla_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu))^\top \Delta\mathrm{z} 
+ \frac{1}{2}\Delta\mathrm{z}^\top \textbf{H}_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu)\Delta\mathrm{z}  \\

\text{s.t.} &\quad g(\mathrm{z}) + \textbf{J}g(\mathrm{z})\Delta\mathrm{z} = 0 \\
&\quad h(\mathrm{z}) + \textbf{J}h(\mathrm{z})\Delta\mathrm{z} \leq 0,
\end{align*}
$$

### Define functions needed for solving the optimization problem

In [ ]:
x_start = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
x_goal  = np.array([3.0, 0.0, 0.0, 0.0, 0.0, 0.0])
u_ref = np.array([m * grav / 2.0, m * grav / 2.0])

Q   = np.diag([10.0, 10.0, 5.0, 5.0, 5.0, 5.0]) * 0.1
Q_F = Q * 5.0
R   = np.diag([0.5, 0.5])


Define
- the cost function $J(\mathrm{z})$
- gradient of the cost function $\nabla_{\mathrm{z}} J(\mathrm{z})$
- Hessian of the cost function $\textbf{H}_{\mathrm{z}} J(\mathrm{z})$

In [ ]:
def unzip_decision_vars(z):
    n_state_vars = (N+1) * x_dim
    X = z[:n_state_vars].reshape(N+1, x_dim)
    U = z[n_state_vars:].reshape(N, u_dim)
    return X, U

def J(z):
    """Cost function"""
    X, U = unzip_decision_vars(z)
    #X_ref = reference_trajectory()

    total = 0.0
    for k in range(N):
        dx = X[k] - x_goal
        total += 0.5 * dx.T @ Q @ dx

    dxN = X[N] - x_goal
    total += 0.5 * dxN.T @ Q_F @ dxN

    for k in range(N):
        du = U[k] - u_ref
        total += 0.5 * du.T @ R @ du

    return total

def grad_J(z):
    """Gradient of the cost function w.r.t. z"""
    X, U = unzip_decision_vars(z)
    J_z = np.zeros_like(z)

    for k in range(N):
        idx = k * x_dim
        J_z[idx:idx + x_dim] = Q @ (X[k] - x_goal)

    idx = N * x_dim
    J_z[idx:idx + x_dim] = Q_F @ (X[N] - x_goal)

    for k in range(N):
        idx = (N + 1) * x_dim + k * u_dim
        J_z[idx:idx + u_dim] = R @ (U[k] - u_ref)

    return J_z

def Hessian_J(z):
    """Hessian of the cost w.r.t. z"""
    H = np.zeros((z.size, z.size))

    for k in range(N):
        idx = k * x_dim
        H[idx:idx + x_dim, idx:idx + x_dim] = Q

    idx = N * x_dim
    H[idx:idx + x_dim, idx:idx + x_dim] = Q_F

    for k in range(N):
        idx = (N + 1) * x_dim + k * u_dim
        H[idx:idx + u_dim, idx:idx + u_dim] = R

    return H

Define
- the equality constraints $g(\mathrm{z})$
- Jacobian of the equality constraints $\textbf{J} g(\mathrm{z})$

In [ ]:
# Equality constraints and Jacobian
def g(z, x_init):
    """Equality constraints"""
    X, U = unzip_decision_vars(z)
    g_val = []
    g_val.append(X[0] - x_init)
    for k in range(N):
        g_val.append(X[k + 1] - X[k] - dt * dynamics(X[k], U[k]))
    return np.concatenate(g_val) # or np.vstack?

def Jacobian_g(z):
    """Jacobian of the equality constraints w.r.t. z"""
    X, U = unzip_decision_vars(z)
    n_z = z.size
    J_g = np.zeros(((N+1) * x_dim, n_z))

    J_g[0:x_dim, 0:x_dim] = np.eye(x_dim)

    for k in range(N):
        x_k = X[k]
        u_k = U[k]

        A_k, B_k = Jacobian_dynamics(x_k, u_k)

        # derivative with respect to x_{k+1}
        J_g[((k+1)*x_dim):((k+2)*x_dim), ((k+1)*x_dim):((k+2)*x_dim)] = np.eye(x_dim)

        # derivative with respect to x_k
        J_g[((k+1)*x_dim):((k+2)*x_dim), (k*x_dim):((k+1)*x_dim)] = -(np.eye(x_dim) + dt * A_k)

        # derivative with respect to u_k
        J_g[((k+1)*x_dim):((k+2)*x_dim), ((N+1)*x_dim+k*u_dim):((N+1)*x_dim+(k+1)*u_dim)] = -(dt * B_k)

    return J_g

Define
- the inequality constraints $h(\mathrm{z})$
- Jacobian of the inequality constraints $\textbf{J} h(\mathrm{z})$

In [ ]:
# Inequality constraints and Jacobian
def h(z, obstacle_center):
    """Inequality constraints"""
    X, U = unzip_decision_vars(z)
    h_val = []

    # Collision avoidance constraints
    for k in range(N+1):
        d = X[k][:2] - obstacle_center
        h_val.append((2.0 * r) ** 2 - d @ d)

    # Control input bounds, written as h(z) <= 0: 0 <= u1, u2 <= 10
    for k in range(N):
        h_val.append(-U[k][0])           # u1 >= 0
        h_val.append(U[k][0] - 10.0)     # u1 <= 10
        h_val.append(-U[k][1])           # u2 >= 0
        h_val.append(U[k][1] - 10.0)     # u2 <= 10

    return np.asarray(h_val, dtype=float)

def Jacobian_h(z, obstacle_center):
    """Jacobian of the inequality constraints w.r.t. z"""
    X, U = unzip_decision_vars(z)
    # Number of inequality constraints: (N+1) collision constraints + 4*N control bounds
    n_h = (N+1) + 4*N
    J_h = np.zeros((n_h, z.size))

    # Jacobian for collision avoidance constraints
    for k in range(N+1):
        grad = np.zeros(x_dim)
        grad[0] = -2.0 * (X[k][0] - obstacle_center[0])
        grad[1] = -2.0 * (X[k][1] - obstacle_center[1])
        J_h[k, (k*x_dim):((k+1)*x_dim)] = grad

    # Jacobian for control input bounds
    for k in range(N):
        u_idx_start = (N+1) * x_dim + k * u_dim
        h_idx = N + 1 + 4*k
        
        # u1 >= 0
        J_h[h_idx, u_idx_start] = -1.0
        # u1 <= 10
        J_h[h_idx + 1, u_idx_start] = 1.0
        # u2 >= 0
        J_h[h_idx + 2, u_idx_start + 1] = -1.0
        # u2 <= 10
        J_h[h_idx + 3, u_idx_start + 1] = 1.0

    return J_h

Define
- gradient of the Lagrangian function $\nabla_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu)$
- Hessian of the Lagrangian function $\textbf{H}_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu)$

In [ ]:
def grad_L(z, lam, nu, obstacle_center):
    """Gradient of the Lagrangian function"""
    return grad_J(z) + Jacobian_g(z).T @ lam + Jacobian_h(z, obstacle_center).T @ nu

def Hessian_L(z, lam, nu, obstacle_center):
    """Hessian of the Lagrangian: approximated by the Hessian of the cost"""
    return Hessian_J(z)

Define a function that checks if a candidate pair $(\widehat{a}, \widehat{b})$ is *dominated* by every pair in
$$
\{ (a_1, b_1), \dotsc, (a_n, b_n) \}.
$$
What it means for $(\widehat{a}, \widehat{b})$ to be dominated by $(a_i, b_i)$ is that $a_i \leq \widehat{a}$ and $b_i \leq \widehat{b}$. See Definition 15.2 of Nocedal and Wright (Numerical Optimization).

In [ ]:
# Define function
def is_dominated(candidate_pair, pairs, verbose=False):
    """
    Returns True if candidate_pair is dominated by every pair in
    pairs, and False otherwise.
    """
    assert(len(candidate_pair) == 3)
    a_hat, b_hat, c_hat = candidate_pair
    for pair in pairs:
        a, b, c = pair
        if (a <= a_hat) and (b <= b_hat) and (c <= c_hat):
            if verbose:
                print(f'({a:6.3e}, {b:6.3e}, {c:6.3e}) dominates ({a_hat:6.3e}, {b_hat:6.3e}, {c_hat:6.3e})')
            return True
    return False

# Test function
assert(is_dominated([10., 1.0, 0.1], [[5., 0., 0.], [15., 0., 0.]], verbose=True))
assert(not is_dominated([1., 0., 0.], [[5., 0., 0.], [15., 0., 0.]], verbose=True))
assert(not is_dominated([1., 1., 1.], [[3., 2., 2.], [2., 3., 3.]], verbose=True))
assert(not is_dominated([1., 4., 4.], [[3., 2., 2.], [2., 3., 3.]], verbose=True))
assert(is_dominated([4., 4., 4.], [[3., 2., 2.], [2., 3., 3.]], verbose=True))

## SQP solver
Recall that the approximated QP problem is given by
$$
\begin{align*}
 \min_{\Delta\mathrm{z}}& \quad (\nabla_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu))^\top \Delta\mathrm{z} 
+ \frac{1}{2}\Delta\mathrm{z}^\top \textbf{H}_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu)\Delta\mathrm{z}  \\

\text{s.t.} &\quad g(\mathrm{z}) + \textbf{J}g(\mathrm{z})\Delta\mathrm{z} = 0 \\
&\quad h(\mathrm{z}) + \textbf{J}h(\mathrm{z})\Delta\mathrm{z} \leq 0,
\end{align*}
$$
where the Lagrangian function is defined as

$$ L(\mathrm{z},\lambda, \nu) = J(\mathrm{z}) + \lambda^\top g(\mathrm{z}) + \nu^\top h(\mathrm{z})$$


Since the SQP is an an alternative to the root-finding Newton step, we will make sure $\textbf{H}_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu)$ is positive definite by adding regularizing term, i.e., $\textbf{H}_{\mathrm{z}} L(\mathrm{z}, \lambda, \nu) + \mu I$.
By solving the QP, we will obtain $\Delta z$ and the corresponding Lagrange multipliers $\lambda^{\text{QP}}$ and $\nu^{\text{QP}}$.
To be equivalent to the root-finding Newton step, we will set $\Delta\lambda \coloneqq \lambda^{\text{QP}}$ and $\Delta\nu \coloneqq \nu^{\text{QP}}$.
Then, we have the following update law 
- $\mathrm{z} \leftarrow \mathrm{z} + \alpha \Delta z$
- $\lambda \leftarrow \lambda + \alpha \Delta\lambda$ 
- $\nu \leftarrow \nu + \alpha \Delta\nu$ 

where $\alpha$ is determined by the filter method. 

In [ ]:
def solve_sqp(z, x_init, obstacle_center, verbose = True):

    z = np.asarray(z, dtype=float).copy()
    z[:x_dim] = x_init

    lam = np.zeros((N+1)*x_dim)
    nu = np.zeros((N+1) + 4*N)

    res = [
        np.linalg.norm(grad_L(z, lam, nu, obstacle_center), np.inf),
        np.linalg.norm(g(z, x_init), np.inf),
        np.linalg.norm(np.maximum(0, h(z, obstacle_center)), np.inf),
    ]
    cost = J(z)

    # Create list of pairs for filter method
    pairs = [[J(z), np.linalg.norm(g(z, x_init), np.inf), np.linalg.norm(np.maximum(0, h(z, obstacle_center)), np.inf)]]

    # Choose parameters
    max_iters = 500
    max_inner_iters = 10
    tol = 1e-5
    rho = 0.5
    delta = 1e-6
    omega = 10.0

    # Iterate
    alpha = None
    H = None
    mu = None
    success = False
    for i in range(max_iters):
        # Show progress
        mu_str = f' ; mu = {mu:.2e}' if mu is not None else ''
        alpha_str = f' ; alpha = {alpha:.2e}' if alpha is not None else ''
        if verbose:
            print(f'{i:3d} : |L_z| = {res[0]:11.8f}; |g| = {res[1]:11.8f}; |h| = {res[2]:11.8f}; J = {cost:7.4f}' + mu_str + alpha_str)

        # Check stopping condition (residuals)
        if (res[0] < tol) and (res[1] < tol) and (res[2] < tol):
            success = True
            if i == 0:
                print(f'success (initial guess satisfies necessary conditions for optimality)')
            else:
                print(f'success (converged at iteration {i})')
            break
        
        # Choose descent direction (Newton step with regularization)
        grad_J_val = grad_J(z)
        Hessian_L_val = Hessian_L(z, lam, nu, obstacle_center)
        #H = Hessian_L_val.copy()
        grad_L_val = grad_L(z, lam, nu, obstacle_center)
        Jacobian_g_val = Jacobian_g(z)
        g_val = g(z, x_init)
        Jacobian_h_val = Jacobian_h(z, obstacle_center)
        h_val = h(z, obstacle_center)
        mu_iters = 0
        while True:
            try:
                # Add regularization to the Hessian
                if mu_iters == 0:
                    mu = 0.0
                else:
                    mu = delta * (omega**mu_iters)
                H = Hessian_L_val + (mu * np.eye(len(z)))
                # Attempt a Cholesky factorization - if it fails, then H is not
                # positive definite and we need to add more regularization
                L_chol = np.linalg.cholesky(H)
                # Attempt to solve for the descent direction - if it fails, then
                # H is ill-conditioned and we need to add more regularization

                #problem = Problem(P=H, q=grad_L_val, A=Jacobian_g_val, b=-g_val, G=Jacobian_h_val, h=-h_val)
                # or
                problem = Problem(P=H, q=grad_J_val, A=Jacobian_g_val, b=-g_val, G=Jacobian_h_val, h=-h_val)
                solution = solve_problem(problem, solver='proxqp', eps_abs=1e-8, eps_rel=1e-8)
                step_z = solution.x
                #step_lam = solution.y
                #step_nu = solution.z
                step_lam = solution.y - lam
                step_nu = solution.z - nu
                #
                # ^^^ Be careful to distinguish between solution.y (dual variable
                #     associated with equality constraints) and solution.z (dual
                #     variable associated with inequality constraints). Get this
                #     wrong and you will have trouble finding your mistake!
                #
                break
            except np.linalg.LinAlgError:
                mu_iters += 1
        
        # Apply backtracking line search (filter method)
        alpha = 0.6
        no_progress = True
        for i_inner in range(max_inner_iters):
            if is_dominated([J(z + alpha * step_z),
                            np.linalg.norm(g(z + alpha * step_z, x_init), np.inf),
                            np.linalg.norm(np.maximum(0, h(z + alpha * step_z, obstacle_center)), np.inf)], 
                            pairs, verbose=False):
                alpha *= rho
            else:
                no_progress = False
                break
        
        # Check stopping condition (no progress)
        if no_progress:
            print(f'failure (no progress at iteration {i})')
            break

        # Update guess
        z = z + alpha * step_z
        z[:x_dim] = x_init
        lam = lam + alpha * step_lam
        nu = nu + alpha * step_nu

        res = [
            np.linalg.norm(grad_L(z, lam, nu, obstacle_center), np.inf),
            np.linalg.norm(g(z, x_init), np.inf),
            np.linalg.norm(np.maximum(0, h(z, obstacle_center)), np.inf),
        ]
        cost = J(z)
        pairs.append([J(z), np.linalg.norm(g(z, x_init), np.inf), np.linalg.norm(np.maximum(0, h(z, obstacle_center)), np.inf)])

    # Check if max iters was exceeded
    if (not success) and (i == max_iters-1):
        print(f'failure (exceeded maximum number {max_iters} of iterations)')

    return z

Test the SQP solver

In [ ]:
# Test the SQP solver
z = np.zeros((N+1)*x_dim + N*u_dim)
z[(N+1)*x_dim:] = np.tile(u_ref, N)
obstacle_center = [1.0, 0.0]

z_opt = solve_sqp(z, x_start, obstacle_center)

plt.figure()
plt.plot(z_opt[:(N+1)*x_dim].reshape(N+1, x_dim)[:, 0], z_opt[:(N+1)*x_dim].reshape(N+1, x_dim)[:, 1], label='optimized trajectory')
circle = plt.Circle(obstacle_center, r, color='red', alpha=0.5)
plt.axis('equal')
plt.xlim(0.0, 1.2)
plt.gca().add_patch(circle)

plt.figure()
plt.plot(z_opt[(N+1)*x_dim:].reshape(N, u_dim)[:, 0], label='T1')
plt.plot(z_opt[(N+1)*x_dim:].reshape(N, u_dim)[:, 1], label='T2')
plt.legend()
plt.show()

## Model-Predictive Control (MPC)

In [ ]:
def run_mpc(obstacle_center, sim_time=sim_time):
    """MPC loop"""

    # Initialization: use hover at (0, 0) as the initial guess trajectory
    z = np.zeros((N+1)*x_dim + N*u_dim)
    z[:(N+1)*x_dim] = np.tile(x_start, N+1)
    z[(N+1)*x_dim:] = np.tile(u_ref, N)

    x = x_start.copy()

    x_hist = []
    u_hist = []

    tt = np.arange(0, sim_time, dt)

    for i in range(len(tt)):
        print("Time: ", tt[i])

        x_hist.append(x.copy())

        # Solve SQP
        z_opt = solve_sqp(z, x.copy(), obstacle_center, verbose=False)
        X_opt, U_opt = unzip_decision_vars(z_opt)

        # Apply the first control only
        u = U_opt[0].copy()
        u_hist.append(u)

        # Propagate the state with zero-order hold on control
        if i < len(tt) - 1:
            t_span = (tt[i], tt[i + 1])
            sol = solve_ivp(lambda t, y: dynamics(y, u),
                            t_span,
                            x,
                            t_eval = [tt[i + 1]],
                            rtol = 1e-9,
                            atol = 1e-9,
            )
            x = sol.y[:, -1]

        # Warm start the next SQP solver
        z = z_opt

    return np.asarray(x_hist), np.asarray(u_hist)

def plot_mpc_result(x_hist, u_hist, obstacle_center):

    tt = np.arange(u_hist.shape[0]) * dt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(x_hist[:, 0], x_hist[:, 1], label='drone trajectory')
    for i in range(x_hist.shape[0]):
        drone = plt.Circle([x_hist[i, 0], x_hist[i, 1]], r, color='blue', alpha=0.05)
        axes[0].add_patch(drone)
    axes[0].scatter([x_start[0], x_goal[0]], [x_start[1], x_goal[1]], c=['green', 'black'], label='start/goal')
    obstacle = plt.Circle(obstacle_center, r, color='red', alpha=0.3, label='obstacle')
    axes[0].add_patch(obstacle)
    axes[0].set_xlabel('$p_x$')
    axes[0].set_ylabel('$p_z$')
    axes[0].grid(True)
    axes[0].axis('equal')
    axes[0].legend()

    axes[1].plot(tt, u_hist[:, 0], label='$T_1$')
    axes[1].plot(tt, u_hist[:, 1], label='$T_2$')
    axes[1].set_xlabel('time (s)')
    axes[1].set_ylabel('Control Inputs')
    axes[1].grid(True)
    axes[1].legend()

    plt.show()


### Implement MPC:

Summary:
- The goal is to get this drone to move from hover at (0,0) to hover at (3,0) while avoiding a single obstacle
- Simulate for a total of 5 seconds
- For first time step, use "hover at (0, 0)" as the initial guess of the trajectory
- For all subsequent time steps, use the optimal trajectory at the prior time step as the initial guess of the trajectory at the current time step (i.e., "warm start" the SQP solver)

A total of four experiments will be carried out:
- Experiment 1: The obstacle is at a location that does not interfere with the drone's flight at all
- Experiment 2: The obstacle is at a location that forces the drone to fly above it
- Experiment 3: The obstacle is at a location that forces the drone to fly below it
- Experiment 4: The obstacle is at a location that prevents the drone from reaching the goal

#### Experiment 1: The obstacle is at a location that does not interfere with the drone's flight at all

In [ ]:
obstacle_center = [4.0, 0.0]

x_hist, u_hist = run_mpc(obstacle_center, sim_time=5.0)

plot_mpc_result(x_hist, u_hist, obstacle_center)

#### Experiment 2: The obstacle is at a location that forces the drone to fly above it

In [ ]:
obstacle_center = [1.5, -0.1]

x_hist, u_hist = run_mpc(obstacle_center, sim_time=5.0)

plot_mpc_result(x_hist, u_hist, obstacle_center)

#### Experiment 3: The obstacle is at a location that forces the drone to fly below it

In [ ]:
obstacle_center = [1.5, 0.15]

x_hist, u_hist = run_mpc(obstacle_center, sim_time=5.0)

plot_mpc_result(x_hist, u_hist, obstacle_center)

#### Experiment 4: The obstacle is at a location that prevents the drone from reaching the goal

In [ ]:
obstacle_center = [3.0, 0.0]

x_hist, u_hist = run_mpc(obstacle_center, sim_time=5.0)

plot_mpc_result(x_hist, u_hist, obstacle_center)

## Gymnasium environment for RL and behavioral cloning

This environment keeps the planar quadrotor state from above. The observation augments that state with the relative goal and obstacle positions, while the action is the two bounded rotor thrusts. The state limits are stored as `(x_dim, 2)` arrays whose rows are `[min, max]` pairs for each state variable.

In [ ]:
import gymnasium as gym
from gymnasium import spaces


DEFAULT_STATE_LIMIT = np.array([
    [-1.0, 4.0],
    [-1.5, 1.5],
    [-np.pi, np.pi],
    [-6.0, 6.0],
    [-6.0, 6.0],
    [-12.0, 12.0],
], dtype=np.float64)

DEFAULT_OBSTACLE_LIMIT = np.array([
    [0.8, 2.5],
    [-0.8, 0.8],
], dtype=np.float64)


class PlanarQuad(gym.Env):
    """Planar quadrotor reaching task with one circular obstacle."""

    metadata = {"render_modes": []}

    def __init__(
        self,
        goal=x_goal,
        dt=dt,
        max_episode_steps=50,
        state_limit=DEFAULT_STATE_LIMIT,
        obstacle_limit=DEFAULT_OBSTACLE_LIMIT,
        dynamics_fn=dynamics,
    ):
        super().__init__()

        self.dt = float(dt)
        self.max_episode_steps = int(max_episode_steps)
        self.goal = np.array(goal, dtype=np.float64, copy=True)
        
        self.dynamics_fn = dynamics_fn

        self.state_limit = np.array(state_limit, dtype=np.float64, copy=True)
        self.obstacle_limit = np.array(obstacle_limit, dtype=np.float64, copy=True)
        
        self.goal_position_tolerance = 0.08
        self.goal_velocity_tolerance = 0.15
        self.goal_angle_tolerance = 0.12
        self.goal_omega_tolerance = 0.20

        self.drone_radius = r
        self.obstacle_radius = r

        self.observation_space = spaces.Box(
            low=np.concatenate([
                self.state_limit[:, 0],
                self.goal[:2] - self.state_limit[:2, 1],
                self.obstacle_limit[:, 0] - self.state_limit[:2, 1],
            ]).astype(np.float32),
            high=np.concatenate([
                self.state_limit[:, 1],
                self.goal[:2] - self.state_limit[:2, 0],
                self.obstacle_limit[:, 1] - self.state_limit[:2, 0],
            ]).astype(np.float32),
            dtype=np.float32,
        )
        self.action_space = spaces.Box(
            low=np.zeros(u_dim, dtype=np.float32),
            high=np.full(u_dim, 10.0, dtype=np.float32),
            dtype=np.float32,
        )

        self.state = None
        self.obstacle_center = None
        self.step_count = 0

    def _get_observation(self):
        goal_rel = self.goal[:2] - self.state[:2]
        obstacle_rel = self.obstacle_center - self.state[:2]
        return np.concatenate([self.state, goal_rel, obstacle_rel]).astype(np.float32)

    def _goal_distance(self):
        return float(np.linalg.norm(self.goal[:2] - self.state[:2]))

    def _obstacle_distance(self):
        return float(np.linalg.norm(self.state[:2] - self.obstacle_center))

    def _state_in_bounds(self, state):
        return bool(np.all(state >= self.state_limit[:, 0]) and np.all(state <= self.state_limit[:, 1]))

    def _in_bounds(self):
        return self._state_in_bounds(self.state)

    def _collision_radius(self):
        return self.drone_radius + self.obstacle_radius

    def _is_success(self):
        return bool(
            self._goal_distance() <= self.goal_position_tolerance
            and np.linalg.norm(self.state[3:5]) <= self.goal_velocity_tolerance
            and abs(self.state[2]) <= self.goal_angle_tolerance
            and abs(self.state[5]) <= self.goal_omega_tolerance
        )

    def _is_collision(self):
        return self._obstacle_distance() <= self._collision_radius()

    def _sample_initial_state(self):
        return self.np_random.uniform(
            self.state_limit[:, 0],
            self.state_limit[:, 1],
        ).astype(np.float64)

    def _sample_obstacle(self):
        return self.np_random.uniform(
            self.obstacle_limit[:, 0],
            self.obstacle_limit[:, 1],
        ).astype(np.float64)

    def _sample_feasible_reset(self):
        min_clearance = self._collision_radius() + 0.15
        for _ in range(1000):
            state = self._sample_initial_state()
            obstacle = self._sample_obstacle()
            start_clear = np.linalg.norm(state[:2] - obstacle) > min_clearance
            goal_clear = np.linalg.norm(self.goal[:2] - obstacle) > min_clearance
            if self._state_in_bounds(state) and start_clear and goal_clear:
                return state, obstacle
        raise RuntimeError("Could not sample a feasible initial state and obstacle")

    def _reward(self, action):

        goal_distance = self._goal_distance()
        attitude_error = abs(self.state[2]) + 0.2 * abs(self.state[5])
        control_effort = np.linalg.norm((action - u_ref) / 10.0) ** 2
        obstacle_clearance = self._obstacle_distance() - self._collision_radius()

        reward = 0.75 * goal_distance
        reward -= 0.05 * attitude_error
        reward -= 0.01 * control_effort
        reward -= 0.01

        if obstacle_clearance < 0.2:
            reward -= 2.0 * (0.2 - obstacle_clearance) ** 2

        if self._is_success():
            reward += 100.0
        elif self._is_collision():
            reward -= 100.0
        elif not self._in_bounds():
            reward -= 50.0

        return float(reward)

    def _get_info(self):
        return {
            "goal": self.goal.copy(),
            "obstacle_center": self.obstacle_center.copy(),
            "step_count": self.step_count,
            "distance_to_goal": self._goal_distance(),
            "distance_to_obstacle": self._obstacle_distance(),
            "success": self._is_success(),
            "collision": self._is_collision(),
            "out_of_bounds": not self._in_bounds(),
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.step_count = 0
        self.state, self.obstacle_center = self._sample_feasible_reset()

        return self._get_observation(), self._get_info()

    def step(self, action):
        action = np.asarray(action, dtype=np.float64)
        action = np.clip(action, self.action_space.low, self.action_space.high)

        previous_goal_distance = self._goal_distance()
        sol = solve_ivp(
            lambda t, y: self.dynamics_fn(y, action),
            (0.0, self.dt),
            self.state,
            t_eval=[self.dt],
            rtol=1e-8,
            atol=1e-10,
        )

        integration_failed = not sol.success
        if sol.y.size:
            self.state = sol.y[:, -1].astype(np.float64)

        self.step_count += 1

        success = self._is_success()
        collision = self._is_collision()
        out_of_bounds = not self._in_bounds()
        terminated = bool(success or collision or out_of_bounds or integration_failed)
        truncated = bool(self.step_count >= self.max_episode_steps and not terminated)

        reward = self._reward(action)

        return self._get_observation(), reward, terminated, truncated, self._get_info()


In [ ]:
# Smoke test: confirm Gymnasium-style rollouts work.
def rollout_random_policy(env, episodes=10, seed=0):
    returns = []
    lengths = []
    final_infos = []

    for episode in range(episodes):
        obs, info = env.reset(seed=seed + episode)
        total_reward = 0.0

        for _ in range(env.max_episode_steps):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break

        returns.append(total_reward)
        lengths.append(info["step_count"])
        final_infos.append(info)
        print(
            f"episode {episode}: steps={info['step_count']}, "
            f"return={total_reward:.2f}, "
            f"goal_dist={info['distance_to_goal']:.2f}, "
            f"obstacle_dist={info['distance_to_obstacle']:.2f}"
        )

    return returns, lengths, final_infos


env = PlanarQuad()
obs, info = env.reset(seed=0)
print("observation shape:", obs.shape)
print("action space:", env.action_space)
print("observation space:", env.observation_space)

rollout_returns, rollout_lengths, rollout_infos = rollout_random_policy(env, episodes=10, seed=1)


observation shape: (10,)
action space: Box(0.0, 10.0, (2,), float32)
observation space: Box([ -1.         -1.5        -3.1415927  -6.         -6.        -12.
  -1.         -1.5        -3.2        -2.3      ], [ 4.         1.5        3.1415927  6.         6.        12.
  4.         1.5        3.5        2.3      ], (10,), float32)
episode 0: steps=1, return=-48.86, goal_dist=1.72, obstacle_dist=1.44
episode 1: steps=1, return=-47.95, goal_dist=2.97, obstacle_dist=1.04
episode 2: steps=3, return=-41.88, goal_dist=3.87, obstacle_dist=2.38
